# 07 - השוואה אזורית

מחברת זו שואלת האם **הפגיעוּת מתפלגת באופן אחיד ברחבי ישראל**. השלבים הקודמים בנו את הגרף ברמת
התחנות, מנו את נקודות החיתוך (articulation points) שלו, והעניקו לכל תחנה ציון באמצעות מדדי
מרכזיות. כאן אנו מצרפים את התוצאות שהתקבלו לכל תחנה לפי **אזור** (צפון / מרכז / דרום / ירושלים,
המשויך לפי קואורדינטות בשלב 01) ולפי **מטרופולין** (תל אביב / חיפה / ירושלים / באר שבע / פריפריה),
ומשווים עד כמה מרוכזת התשתית הקריטית בכל אחד מהם. הפלט הוא טבלת השוואה אחת בתוספת האיורים
הנכללים בדוח.

**שאלת המחקר הנענית כאן:** *האם הנזק הנגרם מהסרת תחנות מפתח מתפלג באופן שונה בין מרכז הארץ
לפריפריה?*

### קלט (שהופק במחברות קודמות)
- `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - שורה אחת לכל תחנה עם
  `stop_id, stop_name, region, metro, lat, lon, degree, betweenness, ...`
- `outputs/nb/03_network_descriptive_analysis/tables/articulation_points.csv` - התחנות שהסרתן
  מנתקת את הגרף
- `outputs/nb/03_network_descriptive_analysis/tables/bridges.csv` - *אופציונלי*; אם אינו קיים,
  עמודות ה-bridges פשוט מדולגות

שמות התיקיות אינם מקודדים קשיחות: הטוען סורק את עץ `outputs/nb` בחיפוש אחר שמות הקבצים ומעדיף
תיקייה ששמה מתחיל במספר השלב הצפוי.

### פלט (הכול תחת `outputs/nb/07_regional_comparison/`)
`tables/`
- `regional_summary.csv` - התוצר המרכזי: שורה אחת לכל אזור
- `metro_summary.csv` - אותו פילוח לפי מטרופולין
- `stops_with_region.csv` - כל התחנות עם הדגלים `is_ap` / `is_critical` שנעשה בהם שימוש כאן
- `top_critical_by_region.csv` - 5 התחנות הקריטיות בעלות ה-betweenness הגבוה ביותר בכל אזור
- `regional_significance.csv` - מבחן chi-square של אזור מול קריטיוּת

`figures/`
- `critical_stations_by_region.png`, `ap_and_bridges_by_region.png`,
  `avg_betweenness_by_region.png`, `stations_map_by_region.png`,
  `regional_vulnerability_comparison.png`, `metro_vulnerability_comparison.png`

**זמן ריצה:** שניות. מחברת זו רק מצרפת קבצי CSV שחושבו כבר בשלבים קודמים - לא רץ כאן שום
אלגוריתם גרפים.

## 1. אתחול סביבת העבודה

מאתר את המאגר (ומשכפל אותו כאשר אנו על Google Colab), קובע את שורש המאגר כספריית העבודה, ויוצר את
תיקיית הפלט המשותפת `outputs/nb`. כל מחברת בפרויקט זה נפתחת בתא זהה זה, כך שכל הסדרה רצה ללא
שינוי הן מקומית והן ב-Colab.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, קבועים ניתנים לכוונון ותיקיית הפלט של השלב

אנו מתקינים ומייבאים את חבילות החישוב המדעי, ולאחר מכן מצהירים על כל פרמטרי הכוונון של הניתוח
במקום אחד, כך שבודק יוכל לשנות את הגדרת "קריטי" ולהריץ מחדש:

- `CRITICAL_QUANTILE = 0.90` - סף העשירון העליון (top-10%) של betweenness, שבו נעשה שימוש לאורך כל
  הפרויקט.
- `MIN_REGION_N = 100` - כל קבוצה שמספר התחנות בה קטן מערך זה מסומנת כמדגם קטן, משום שאחוז המחושב
  על קומץ תחנות הוא רעש ולא ממצא.
- `WILSON_Z = 1.96` - ערך ה-z עבור רווח הסמך של 95% המשורטט על כל שיעור.

השלב כותב אך ורק לתוך התיקייה שלו, `outputs/nb/07_regional_comparison/`.

In [ ]:
_ensure("pandas", "numpy", "matplotlib", "seaborn", "scipy")

import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
pd.set_option("display.width", 160)

# ---- Tunables -------------------------------------------------------------------
CRITICAL_QUANTILE = 0.90   # "critical" = betweenness in the top 10% of the whole network
MIN_REGION_N      = 100    # below this many stops, a group is flagged as small-sample
WILSON_Z          = 1.96   # 95% confidence interval on every reported share

# ---- Stage folders --------------------------------------------------------------
STAGE   = OUT / "07_regional_comparison"
TABLES  = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Stage folder:", STAGE)

## 3. טקסט בעברית באיורים

הזנת ה-GTFS היא ישראלית, ולכן שמות התחנות, שמות האזורים ושמות המטרופולינים הם בעברית. Matplotlib
אינו מיישם את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן העברית מוצגת הפוכה. אנו מתקנים
את `matplotlib.text.Text.set_text` פעם אחת, לפני כל שרטוט. שמות אזורים ומטרופולינים אף מתורגמים
לאנגלית עבור האיורים (ראו מפות התוויות להלן), אך כל ערך שלא צפינו מראש נסוג לעברית בסדר תווים
תקין במקום לג'יבריש.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. תוויות, צבעים וסדר עקביים

פרט קטן אך חשוב עבור דוח: אותו אזור חייב לקבל את אותו צבע בכל איור. הסקריפט המקורי צבע את העמודות
לפי *מיקום*, ולכן לאחר מיון אזור החליף צבע בין תרשימים - אנו מתקנים זאת באמצעות מיפוי מפורש משם
לצבע. האזורים משורטטים תמיד בסדר הקבוע מרכז, צפון, דרום, ירושלים; כל ערך בלתי צפוי מתווסף בסוף
בצבע אפור.

In [ ]:
REGION_ORDER = ["מרכז", "צפון", "דרום", "ירושלים"]
REGION_EN = {"מרכז": "Center", "צפון": "North",
             "דרום": "South", "ירושלים": "Jerusalem"}
REGION_COLORS = {"מרכז": "#2563eb", "צפון": "#16a34a",
                 "דרום": "#dc2626", "ירושלים": "#d97706"}

METRO_EN = {"תל אביב": "Tel Aviv", "חיפה": "Haifa", "ירושלים": "Jerusalem",
            "באר שבע": "Be'er Sheva", "פריפריה": "Periphery"}
METRO_COLORS = {"תל אביב": "#2563eb", "חיפה": "#0891b2", "ירושלים": "#d97706",
                "באר שבע": "#dc2626", "פריפריה": "#6b7280"}

GREY = "#6b7280"

def region_label(name):
    """English label for a region, falling back to display-ordered Hebrew."""
    return REGION_EN.get(name, fix_he(name))

def metro_label(name):
    return METRO_EN.get(name, fix_he(name))

def region_color(name):
    return REGION_COLORS.get(name, GREY)

def metro_color(name):
    return METRO_COLORS.get(name, GREY)

def in_region_order(values):
    """Sort region names into the canonical order; unknown names go last, alphabetically."""
    known = [r for r in REGION_ORDER if r in set(values)]
    rest = sorted(v for v in set(values) if v not in REGION_ORDER)
    return known + rest

def annotate_bars(ax, bars, texts, pad=0.02, fontsize=9):
    """Write a short label (we use the group size n) just above each bar."""
    top = ax.get_ylim()[1]
    for bar, txt in zip(bars, texts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + top * pad,
                txt, ha="center", va="bottom", fontsize=fontsize, color="#334155")

print("label helpers ready")

## 5. טעינת התוצרים משלבים קודמים

שלב זה צורך תוצאות ואינו מחשב אותן מחדש: טבלת המרכזיות ברמת התחנה מגיעה ממחברת 04, ורשימת נקודות
החיתוך ממחברת 03. `find_artifact` סורק את כל עץ `outputs/nb` בחיפוש אחר שם קובץ ומעדיף את התיקייה
ששמה מתחיל במספר השלב הצפוי, כך ששינוי שם בשלב קודם מפיק הודעה מועילה במקום `FileNotFoundError`
בעומק הניתוח. תיקיית השלב שלנו מוחרגת מן החיפוש, כך שהרצה חוזרת לעולם אינה יכולה לקרוא את הפלט של
עצמה.

`bridges.csv` מטופל כקובץ אופציונלי - הוא מוסיף עמודה תיאורית אחת בלבד.

In [ ]:
def find_artifact(filename, prefer_prefix, produced_by):
    """Locate a file written by an earlier notebook stage under outputs/nb."""
    candidates = sorted({p for p in OUT.rglob(filename) if STAGE not in p.parents})
    if not candidates:
        raise FileNotFoundError(
            f"{filename} not found anywhere under {OUT} - "
            f"run notebook {produced_by} first, it writes this file.")
    preferred = [p for p in candidates
                 if p.relative_to(OUT).parts[0].startswith(prefer_prefix)]
    chosen = (preferred or candidates)[0]
    if len(candidates) > 1:
        print(f"  note: found {len(candidates)} copies of {filename}; using {chosen}")
    return chosen

METRICS_PATH = find_artifact("stop_metrics.csv", "04", "04_centrality_analysis")
AP_PATH = find_artifact("articulation_points.csv", "03", "03_network_descriptive_analysis")

metrics = pd.read_csv(METRICS_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")
ap_df = pd.read_csv(AP_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")

try:
    BRIDGES_PATH = find_artifact("bridges.csv", "03", "03_network_descriptive_analysis")
    bridges = pd.read_csv(BRIDGES_PATH,
                          dtype={"from_stop": str, "to_stop": str},
                          encoding="utf-8-sig")
except FileNotFoundError:
    BRIDGES_PATH, bridges = None, None
    print("  bridges.csv not available - the bridge columns will be skipped.")

# Notebook 04 exports the sampled estimate under the name `approx_betweenness`
# (the name records that it is a k-sample estimate, not exact betweenness).
# Alias it so this notebook works with either schema.
if "betweenness" not in metrics.columns and "approx_betweenness" in metrics.columns:
    metrics["betweenness"] = metrics["approx_betweenness"]
    print("  note: aliased 'approx_betweenness' -> 'betweenness' (sampled estimate)")

# Fail loudly and early if the upstream schema is not what we expect.
for col in ("stop_id", "region", "metro", "degree", "betweenness", "lat", "lon"):
    if col not in metrics.columns:
        raise KeyError(f"'{col}' missing from {METRICS_PATH.name}. "
                       f"Columns present: {list(metrics.columns)}")

for col in ("degree", "betweenness", "lat", "lon"):
    metrics[col] = pd.to_numeric(metrics[col], errors="coerce")
metrics["degree"] = metrics["degree"].fillna(0)
metrics["betweenness"] = metrics["betweenness"].fillna(0)
for col in ("region", "metro"):
    metrics[col] = metrics[col].fillna("").replace("", "Unknown")

print(f"stops loaded      : {len(metrics):,}  <- {METRICS_PATH}")
print(f"articulation pts  : {len(ap_df):,}  <- {AP_PATH}")
print(f"bridges           : {0 if bridges is None else len(bridges):,}")
print("stops per region  :")
print(metrics["region"].value_counts().to_string())

## 6. הגדרת "קריטי" שבה נעשה שימוש כאן

זוהי הבחירה המתודולוגית החשובה ביותר במחברת, ולכן אנו מנסחים אותה במפורש.

> **תחנה היא *קריטית* אם מרכזיות ה-betweenness שלה נמצאת בעשירון העליון (top 10%) של הרשת כולה**
> (`betweenness >= p90`, כאשר הסף מחושב פעם אחת על **כל** התחנות, ולא בנפרד לכל אזור).

זוהי בדיוק ההגדרה שבה נעשה שימוש ב**מחברת 04**, שם נבדקת אותה קבוצת top-10% של betweenness לשאלה
האם קיימת לה חלופה בהליכה ברגל. שמירה על הגדרה זהה מבטיחה שהספירות במחברת זו מתיישבות עם הספירות
שם.

שתי השלכות מכוונות:

- **הסף הוא גלובלי.** אילו לכל אזור היה p90 משלו, כל אזור היה קריטי בדיוק ב-10% מעצם ההגדרה
  וההשוואה הייתה חסרת משמעות. סף גלובלי הוא מה שמאפשר לנו לומר "אזור X מחזיק ביותר מחלקו היחסי
  בתחנות הקריטיות של הרשת".
- **נקודות חיתוך מדווחות בנפרד ואינן ממוזגות פנימה.** נקודת חיתוך היא נקודת כשל *מבנית* (הסרתה
  מנתקת את הגרף), וזוהי תפיסה שונה ומחמירה יותר מאשר עומס תנועה גבוה. הסקריפט המקורי ביצע OR בין
  השתיים; אנו שומרים על הדגל הראשי `is_critical` תואם למחברת 04 וחושפים את האיחוד בעמודה משנית
  בעלת שם מפורש, `is_critical_broad`, כך ששתי הקריאות זמינות מבלי לבלבל ביניהן.

הסתייגות אחת שאנו בודקים בקוד: betweenness מחושב על הרכיב הקשיר הגדול ביותר בלבד, ולכן תחנות
מבודדות מקבלות ציון 0. אם יותר מ-10% מהתחנות שוות בערכן לערך הסף, השוואת ה-`>=` בוחרת בשקט יותר
מ-10% מהרשת - התא מדפיס את השיעור בפועל כדי שהדבר יהיה גלוי.

In [ ]:
ap_set = set(ap_df["stop_id"].astype(str))
metrics["is_ap"] = metrics["stop_id"].astype(str).isin(ap_set)

BTW_THRESHOLD = float(metrics["betweenness"].quantile(CRITICAL_QUANTILE))
metrics["is_critical"] = metrics["betweenness"] >= BTW_THRESHOLD
metrics["is_critical_broad"] = metrics["is_critical"] | metrics["is_ap"]

n_total = len(metrics)
n_crit = int(metrics["is_critical"].sum())
n_ap = int(metrics["is_ap"].sum())
n_both = int((metrics["is_critical"] & metrics["is_ap"]).sum())

print(f"betweenness p{CRITICAL_QUANTILE*100:.0f} threshold : {BTW_THRESHOLD:.8f}")
print(f"critical (top betweenness)   : {n_crit:,}  ({100*n_crit/n_total:.1f}% of all stops)")
print(f"articulation points matched  : {n_ap:,}  ({100*n_ap/n_total:.1f}%)")
print(f"both critical and AP         : {n_both:,}")
print(f"union (is_critical_broad)    : {int(metrics['is_critical_broad'].sum()):,}")

if BTW_THRESHOLD <= 0:
    print("\nWARNING: the p90 threshold is 0, meaning more than 10% of stops have zero "
          "betweenness. Every zero-betweenness stop is being counted as critical - "
          "raise CRITICAL_QUANTILE or restrict to the largest component before trusting this.")
if n_ap == 0:
    print("\nWARNING: no articulation point matched a stop_id in stop_metrics.csv - "
          "the two upstream files may use different id formats.")

## 7. Bridges הנוגעים בכל אזור

*bridge* היא קשת שהסרתה מנתקת את הגרף - המקבילה ברמת הקשת לנקודת חיתוך, ומדד ישיר לכך ש"אזור זה
תלוי בקו יחיד". הסקריפט המקורי טען את `bridges.csv` ומעולם לא עשה בו שימוש; כאן אנו אכן משתמשים בו.

כלל הספירה: כל bridge נספר **פעם אחת לכל אזור נבדל שבו הוא נוגע**. bridge פנימי לאזור אחד מוסיף 1
לאותו אזור; bridge החוצה גבול אזורי מוסיף 1 לכל אחד מהשניים. נקודות קצה שהתחנה שלהן אינה מופיעה
בטבלת המדדים (מצב שאינו אמור להתרחש) מושמטות. התא אינו מבצע דבר אם `bridges.csv` לא נמצא.

In [ ]:
bridge_counts = None
if bridges is not None and len(bridges) and {"from_stop", "to_stop"} <= set(bridges.columns):
    region_of = metrics.drop_duplicates("stop_id").set_index("stop_id")["region"]
    touched = []
    for a, b in zip(bridges["from_stop"].map(region_of), bridges["to_stop"].map(region_of)):
        touched.extend({r for r in (a, b) if isinstance(r, str)})
    bridge_counts = pd.Series(touched, dtype="object").value_counts()
    print("bridges touching each region:")
    print(bridge_counts.to_string())
else:
    print("no bridge data - skipping the bridge columns")

## 8. טבלת הסיכום האזורית

ליבת האגרגציה. עבור כל אזור אנו מדווחים את הספירות הגולמיות (`total_stops`, `critical_stops`,
`ap_stops`), את הממוצעים (`avg_degree`, `avg_betweenness`, `max_betweenness`), והשימושי ביותר
לצורכי השוואה - את **השיעורים היחסיים**: אזור בעל 13 אלף תחנות ואזור בעל 3.7 אלף תחנות אינם ניתנים
להשוואה על בסיס ספירות בלבד.

שתי תוספות ביחס לסקריפט המקורי:

- **רווח סמך Wilson של 95%** על `pct_critical`. רווח Wilson מתנהג באופן סביר עבור קבוצות קטנות
  ועבור שיעורים הקרובים ל-0 או ל-100%, מקום שבו רווח הסמך הנורמלי הסטנדרטי אינו מתנהג כך. זהו
  המדד המלמד אותנו האם שני אזורים אכן נבדלים זה מזה או רק נראים שונים.
- **דגל `small_sample`** (`total_stops < MIN_REGION_N`). כל שורה מסומנת מודפסת כאזהרה ומסומנת
  בהצללה (hatch) באיורים; אין לצטט את האחוז שלה בדוח.

באותו פונקציית עזר נעשה שימוש חוזר עבור פילוח המטרופולינים בתא הבא.

In [ ]:
def wilson_ci(k, n, z=WILSON_Z):
    """95% Wilson score interval (in percent) for k successes out of n."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (100 * max(0.0, center - half), 100 * min(1.0, center + half))

def summarize(df, key):
    """Aggregate stop-level flags into one row per group (region or metro)."""
    s = df.groupby(key).agg(
        total_stops=("stop_id", "count"),
        critical_stops=("is_critical", "sum"),
        critical_broad_stops=("is_critical_broad", "sum"),
        ap_stops=("is_ap", "sum"),
        avg_degree=("degree", "mean"),
        avg_betweenness=("betweenness", "mean"),
        max_betweenness=("betweenness", "max"),
    ).reset_index()
    for c in ("critical_stops", "critical_broad_stops", "ap_stops"):
        s[c] = s[c].astype(int)
    s["pct_critical"] = (100 * s["critical_stops"] / s["total_stops"]).round(2)
    s["pct_critical_broad"] = (100 * s["critical_broad_stops"] / s["total_stops"]).round(2)
    s["pct_ap"] = (100 * s["ap_stops"] / s["total_stops"]).round(2)
    # share of the network's critical stops that sit in this group
    s["share_of_all_critical"] = (100 * s["critical_stops"] / max(1, n_crit)).round(1)
    s["share_of_all_stops"] = (100 * s["total_stops"] / n_total).round(1)
    ci = [wilson_ci(k, n) for k, n in zip(s["critical_stops"], s["total_stops"])]
    s["pct_critical_lo"] = [round(lo, 2) for lo, _ in ci]
    s["pct_critical_hi"] = [round(hi, 2) for _, hi in ci]
    s["small_sample"] = s["total_stops"] < MIN_REGION_N
    s["avg_degree"] = s["avg_degree"].round(3)
    s["avg_betweenness"] = s["avg_betweenness"].round(8)
    return s.sort_values("pct_critical", ascending=False).reset_index(drop=True)

region_summary = summarize(metrics, "region")
if bridge_counts is not None:
    region_summary["bridges_touching"] = (region_summary["region"]
                                          .map(bridge_counts).fillna(0).astype(int))
    region_summary["bridges_per_1000_stops"] = (
        1000 * region_summary["bridges_touching"] / region_summary["total_stops"]).round(2)

region_summary.to_csv(TABLES / "regional_summary.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(TABLES / "stops_with_region.csv", index=False, encoding="utf-8-sig")

print(region_summary.to_string(index=False))

flagged = region_summary[region_summary["small_sample"]]
if len(flagged):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops) - do not quote these percentages:")
    print(flagged[["region", "total_stops", "pct_critical"]].to_string(index=False))
else:
    print(f"\nNo region falls below {MIN_REGION_N} stops - all regional shares are on "
          "thousands of stops and are statistically stable.")
print(f"\nsaved -> {TABLES / 'regional_summary.csv'}")

## 9. אותו פילוח לפי מטרופולין

ארבעת האזורים הם רצועות גיאוגרפיות החתוכות לפי קו רוחב, מה שמאגד יחד עיר צפופה עם השטחים
החקלאיים שסביבה. התווית `metro` משלב 01 מספקת מבט משלים: תחנות בתוך רדיוס קבוע מתל אביב / חיפה /
ירושלים / באר שבע, וכל היתר מתויג כפריפריה. כאן דגל המדגם הקטן עשוי אכן להידלק, ולכן אנו מדפיסים
אותו שוב.

In [ ]:
metro_summary = summarize(metrics, "metro")
metro_summary.to_csv(TABLES / "metro_summary.csv", index=False, encoding="utf-8-sig")
print(metro_summary.to_string(index=False))

flagged_metro = metro_summary[metro_summary["small_sample"]]
if len(flagged_metro):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops):")
    print(flagged_metro[["metro", "total_stops", "pct_critical",
                         "pct_critical_lo", "pct_critical_hi"]].to_string(index=False))
else:
    print(f"\nNo metro area falls below {MIN_REGION_N} stops.")
print(f"\nsaved -> {TABLES / 'metro_summary.csv'}")

## 10. האם ההבדל האזורי אמיתי, או רק נראה לעין?

תרשימי עמודות תמיד נראים שונים. מבחן chi-square לאי-תלות על הטבלה הדו-ממדית *region x is_critical*
שואל האם שיעור התחנות הקריטיות תלוי כלל באזור, ו-**Cramer's V** ממיר את ה-chi-square לגודל אפקט
בין 0 ל-1 שאינו גדל עם גודל המדגם.

יש לקרוא את שניהם יחד ובספקנות: עם עשרות אלפי תחנות, *כל* הבדל יוצא "מובהק ביותר" (p הוא כאן
למעשה פונקציה של n). Cramer's V ורווחי הסמך מהתא הקודם הם המדדים הכנים למידה שבה האזורים אכן
נבדלים זה מזה.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(metrics["region"], metrics["is_critical"])
chi2, p_value, dof, expected = chi2_contingency(contingency)
n_obs = int(contingency.values.sum())
cramers_v = float(np.sqrt(chi2 / (n_obs * (min(contingency.shape) - 1))))
spread = float(region_summary["pct_critical"].max() - region_summary["pct_critical"].min())

sig = pd.DataFrame([{
    "test": "chi-square independence (region x is_critical)",
    "chi2": round(chi2, 2),
    "dof": int(dof),
    "p_value": p_value,
    "n": n_obs,
    "cramers_v": round(cramers_v, 4),
    "pct_critical_spread_points": round(spread, 2),
    "min_expected_count": round(float(expected.min()), 1),
}])
sig.to_csv(TABLES / "regional_significance.csv", index=False, encoding="utf-8-sig")
print(sig.T.to_string(header=False))

print("\nInterpretation:")
print(f"  p = {p_value:.3g} -> the regions do differ, but at n={n_obs:,} that was almost "
      "guaranteed.")
print(f"  Cramer's V = {cramers_v:.3f} -> "
      + ("negligible" if cramers_v < 0.1 else "small" if cramers_v < 0.3
         else "moderate" if cramers_v < 0.5 else "large") + " association.")
print(f"  spread between the highest and lowest region: {spread:.1f} percentage points.")

## 11. איור 1 - תחנות קריטיות לפי אזור

שני פאנלים, משום שהם עונים על שתי שאלות שונות. **משמאל**: המספר המוחלט של תחנות קריטיות, הנשלט
על ידי האזור שפשוט מכיל את מספר התחנות הגדול ביותר. **מימין**: *השיעור* מתוך תחנותיו של כל אזור
שהן קריטיות, עם רווח Wilson של 95% כשגיאת תקן - זהו הפאנל שעונה על השאלה "האם הפריפריה שברירית
יותר?". גודל הקבוצה `n` מודפס מעל כל עמודה, ואזור בעל מדגם קטן היה מצויר בהצללה.

In [ ]:
reg = region_summary.set_index("region").loc[in_region_order(region_summary["region"])].reset_index()
labels = [region_label(r) for r in reg["region"]]
colors = [region_color(r) for r in reg["region"]]
hatches = ["//" if flag else "" for flag in reg["small_sample"]]
n_texts = [f"n={int(v):,}" for v in reg["total_stops"]]

yerr = np.vstack([
    np.clip(reg["pct_critical"] - reg["pct_critical_lo"], 0, None),
    np.clip(reg["pct_critical_hi"] - reg["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

bars = axes[0].bar(labels, reg["critical_stops"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Critical stops per region (count)")
axes[0].set_ylabel("number of critical stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], bars, n_texts)

bars2 = axes[1].bar(labels, reg["pct_critical"], color=colors, edgecolor="white",
                    hatch=hatches, yerr=yerr, capsize=5, ecolor="#334155")
axes[1].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2,
                label=f"network average ({100*n_crit/n_total:.1f}%)")
axes[1].set_title("Share of the region's own stops that are critical")
axes[1].set_ylabel("% of the region's stops")
axes[1].margins(y=0.20)
axes[1].legend(fontsize=9)
annotate_bars(axes[1], bars2, n_texts)

fig.suptitle(f"Critical = betweenness in the network-wide top {100*(1-CRITICAL_QUANTILE):.0f}%"
             "   (error bars: 95% Wilson CI)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "critical_stations_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "critical_stations_by_region.png")

## 12. איור 2 - נקודות כשל מבניות לפי אזור

בעוד שאיור 1 מדד *עומס*, איור זה מודד *מבנה*: שיעור התחנות בכל אזור שהן נקודות חיתוך, וכן (כאשר
`bridges.csv` זמין) כמה bridges נוגעים באזור לכל 1,000 תחנות. אזור יכול לשאת תנועה מתונה ועדיין
להיות שברירי אם הרשת שלו היא שרשרת של קשרים בודדים, וזה בדיוק מה ששתי עמודות אלה חושפות.

In [ ]:
has_bridges = "bridges_per_1000_stops" in reg.columns
ncols = 2 if has_bridges else 1
fig, axes = plt.subplots(1, ncols, figsize=(6.5 * ncols, 5))
axes = np.atleast_1d(axes)

b = axes[0].bar(labels, reg["pct_ap"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Articulation points as % of the region's stops")
axes[0].set_ylabel("% of the region's stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

if has_bridges:
    b2 = axes[1].bar(labels, reg["bridges_per_1000_stops"], color=colors,
                     edgecolor="white", hatch=hatches)
    axes[1].set_title("Bridges touching the region per 1,000 stops")
    axes[1].set_ylabel("bridges per 1,000 stops")
    axes[1].margins(y=0.15)
    annotate_bars(axes[1], b2, n_texts)

plt.tight_layout()
plt.savefig(FIGURES / "ap_and_bridges_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "ap_and_bridges_by_region.png")

## 13. איור 3 - כיצד מתפלג העומס בתוך כל אזור

ממוצעים מסתירים את צורת ההתפלגות, ולכן אנו מציגים שני דברים. **משמאל**: ממוצע ה-betweenness לכל
אזור - כמה תנועה של מסלולים קצרים ביותר נושאת תחנה טיפוסית. **מימין**: התפלגות ה-betweenness בתוך
כל אזור בסקאלה לוגריתמית (ללא אפסים, מאחר ש-log(0) אינו מוגדר), החושפת האם באזור מצויים מספר
מוקדים קיצוניים או מישור רחב. הקו המקווקו הוא סף ה-p90 הגלובלי, כלומר הסף שמעליו תחנה נחשבת
קריטית.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

b = axes[0].bar(labels, reg["avg_betweenness"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Mean betweenness centrality per region")
axes[0].set_ylabel("mean betweenness")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

order = list(reg["region"])
data = [metrics.loc[(metrics["region"] == r) & (metrics["betweenness"] > 0), "betweenness"].values
        for r in order]
parts = axes[1].boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch, c in zip(parts["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.65)
axes[1].set_yscale("log")
axes[1].axhline(max(BTW_THRESHOLD, 1e-12), color="#334155", ls="--", lw=1.2,
                label=f"critical threshold (p{CRITICAL_QUANTILE*100:.0f})")
axes[1].set_title("Betweenness distribution, non-zero stops only (log scale)")
axes[1].set_ylabel("betweenness")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "avg_betweenness_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "avg_betweenness_by_region.png")

## 14. איור 4 - התמונה הגיאוגרפית

כל תחנה משורטטת בקואורדינטות שלה, צבועה לפי אזור, כאשר התחנות הקריטיות מוצגות מעליה בשחור. זוהי
בדיקת שפיות לא פחות מאשר תוצאה: הגדרת האזורים כרצועות קו רוחב משלב 01 אמורה להיראות כחתכים
אופקיים נקיים, והתחנות הקריטיות אמורות לשרטט את הצירים הבין-עירוניים ולא להתפזר באקראי. תחנות
ללא קואורדינטות מושמטות, ומספר התחנות שהושמטו מודפס.

In [ ]:
geo = metrics.dropna(subset=["lat", "lon"])
print(f"stops without coordinates (dropped from the map): {len(metrics) - len(geo):,}")

fig, ax = plt.subplots(figsize=(8, 11))
for r in in_region_order(geo["region"]):
    grp = geo[geo["region"] == r]
    ax.scatter(grp["lon"], grp["lat"], s=2, alpha=0.30, color=region_color(r),
               label=f"{region_label(r)} (n={len(grp):,})")

crit_geo = geo[geo["is_critical"]]
ax.scatter(crit_geo["lon"], crit_geo["lat"], s=14, color="black", alpha=0.70, zorder=5,
           label=f"critical (n={len(crit_geo):,})")

ax.set_title("Stops by region, critical stops in black")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect(1 / np.cos(np.radians(float(geo["lat"].mean()))))
ax.legend(markerscale=4, fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES / "stations_map_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "stations_map_by_region.png")

## 15. איור 5 - השוואה אזורית בארבעה פאנלים

איור הסיכום עבור הדוח: ארבעת המדדים זה לצד זה על אותה קבוצת אזורים - אחוז קריטיות, אחוז נקודות
חיתוך, דרגה ממוצעת (עד כמה מקושרת תחנה טיפוסית), והשיעור מתוך *סך התחנות הקריטיות של הרשת* המצוי
בכל אזור. הפאנל האחרון הוא מבט הריכוזיות: אזור המחזיק בשיעור גדול בהרבה מהתחנות הקריטיות מאשר
משיעורו בתחנות בכלל הוא המקום שבו שיבוש בקנה מידה ארצי יכאב ביותר, ולכן אנו משרטטים את שיעורו מכלל
התחנות כסמן ייחוס.

In [ ]:
panels = [
    ("pct_critical", "% critical stops", "% of the region's stops"),
    ("pct_ap", "% articulation points", "% of the region's stops"),
    ("avg_degree", "Mean degree", "neighbouring stops"),
    ("share_of_all_critical", "Share of ALL critical stops", "% of the network's critical stops"),
]

fig, axes = plt.subplots(1, 4, figsize=(19, 5))
for ax, (col, title, ylab) in zip(axes, panels):
    bars = ax.bar(labels, reg[col], color=colors, edgecolor="white", hatch=hatches)
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.margins(y=0.18)
    annotate_bars(ax, bars, n_texts, fontsize=8)
    if col == "share_of_all_critical":
        ax.scatter(labels, reg["share_of_all_stops"], color="black", marker="_", s=400,
                   zorder=6, label="share of all stops")
        ax.legend(fontsize=8)

fig.suptitle("Regional comparison of network vulnerability (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "regional_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "regional_vulnerability_comparison.png")

## 16. איור 6 - אותה השוואה לפי מטרופולין

חוזר על איור 5 עבור פילוח המטרופולינים, שהוא הניגוד החד יותר: ארבעה אזורים עירוניים צפופים אל מול
כל היתר ("Periphery"). הקבוצות ממוינות לפי שיעור הקריטיות שלהן, `n` מסומן, וכל קבוצה שמספר
התחנות בה נמוך מ-`MIN_REGION_N` מסומנת בהצללה, כך שאחוז גבוה מדומה לא ייקרא כממצא.

In [ ]:
mt = metro_summary.copy()
m_labels = [metro_label(m) for m in mt["metro"]]
m_colors = [metro_color(m) for m in mt["metro"]]
m_hatch = ["//" if f else "" for f in mt["small_sample"]]
m_texts = [f"n={int(v):,}" for v in mt["total_stops"]]

m_yerr = np.vstack([
    np.clip(mt["pct_critical"] - mt["pct_critical_lo"], 0, None),
    np.clip(mt["pct_critical_hi"] - mt["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

b0 = axes[0].bar(m_labels, mt["pct_critical"], color=m_colors, edgecolor="white",
                 hatch=m_hatch, yerr=m_yerr, capsize=4, ecolor="#334155")
axes[0].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2)
axes[0].set_title("% critical stops (95% Wilson CI)")
axes[0].set_ylabel("% of the area's stops")
axes[0].margins(y=0.20)
annotate_bars(axes[0], b0, m_texts, fontsize=8)

b1 = axes[1].bar(m_labels, mt["pct_ap"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[1].set_title("% articulation points")
axes[1].set_ylabel("% of the area's stops")
axes[1].margins(y=0.18)
annotate_bars(axes[1], b1, m_texts, fontsize=8)

b2 = axes[2].bar(m_labels, mt["avg_degree"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[2].set_title("Mean degree")
axes[2].set_ylabel("neighbouring stops")
axes[2].margins(y=0.18)
annotate_bars(axes[2], b2, m_texts, fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Metropolitan comparison (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "metro_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "metro_vulnerability_comparison.png")

## 17. אילו תחנות מניעות בפועל את המספר של כל אזור

אחוזים הם מופשטים; נקיבה בשמות התחנות הופכת את התוצאה לניתנת לבדיקה מול המציאות. עבור כל אזור אנו
מפרטים את חמש התחנות הקריטיות בעלות ה-betweenness הגבוה ביותר, יחד עם הציון האם כל אחת מהן היא גם
נקודת חיתוך. אם שמות אלה הם מוקדים בין-עירוניים מוכרים, הרי שהצינור מודד משהו אמיתי.

In [ ]:
cols = ["region", "metro", "stop_id", "stop_name", "degree", "betweenness", "is_ap"]
cols = [c for c in cols if c in metrics.columns]

top_by_region = (metrics[metrics["is_critical"]]
                 .sort_values("betweenness", ascending=False)
                 .groupby("region", group_keys=False)
                 .head(5)[cols]
                 .sort_values(["region", "betweenness"], ascending=[True, False]))

top_by_region.to_csv(TABLES / "top_critical_by_region.csv", index=False, encoding="utf-8-sig")
print(top_by_region.to_string(index=False))
print(f"\nsaved -> {TABLES / 'top_critical_by_region.csv'}")

print("\nAll artifacts written by this notebook:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT))

## מסקנות

*(המספרים המדויקים מודפסים על ידי התאים שלעיל ונשמרים בקובץ
`outputs/nb/07_regional_comparison/tables/regional_summary.csv`; ההיגדים שלהלן מתארים את התבנית
שטבלאות אלה מציגות.)*

1. **הפגיעוּת אינה מתפלגת באופן אחיד, אך הפער מתון.** תחת ההגדרה של הפרויקט (קריטי = עשירון עליון
   של betweenness ברמת הרשת כולה), לאזור המרכז יש השיעור *הנמוך ביותר* של תחנות קריטיות, בעוד
   שהצפון, הדרום וירושלים נמצאים כולם מעל לממוצע הרשת של 10%. הכיוון תומך בהשערת "הפריפריה שברירית
   יותר", אך הפער הוא של מספר נקודות אחוז - לא של סדר גודל.

2. **האות המבני חזק מן האות התנועתי.** שיעור התחנות שהן נקודות חיתוך נבדל בין האזורים בפקטור יחסי
   גדול יותר מאשר שיעור התחנות הקריטיות, כאשר ירושלים גבוהה בבירור והמרכז נמוך ביותר. רשתות
   סבוכות וצפופות מכילות יתירות; רשתות דלילות מנתבות את הכול דרך תחנות בודדות. זוהי הראיה המשכנעת
   יותר לטענת אי-השוויון האזורי.

3. **במונחים מוחלטים המרכז עדיין שולט.** הוא מחזיק בהרבה יותר תחנות קריטיות מכל אזור אחר, פשוט
   משום שהוא מחזיק בהרבה יותר תחנות. ניתוח שיבוש בקנה מידה ארצי צריך לקרוא את פאנל הספירות, וניתוח
   הוגנוּת צריך לקרוא את פאנל האחוזים - הם מצביעים לכיוונים מנוגדים, ודיווח על אחד מהם בלבד היה
   מטעה.

4. **המובהקות הסטטיסטית כאן כמעט חסרת משמעות; גודל האפקט אינו כזה.** עם כ-30 אלף תחנות, ערך ה-p של
   מבחן ה-chi-square קטן באופן אסטרונומי, ואף על פי כן Cramer's V יוצא נמוך מאוד. קריאה כנה:
   ההבדלים האזוריים *אמיתיים אך חלשים*. רווחי Wilson של 95% באיורים צרים רק משום שכל אזור מכיל
   אלפי תחנות.

5. **אף אזור אינו מדגם קטן; יש לבדוק במקום זאת את טבלת המטרופולינים.** כל ארבעת האזורים מחזיקים
   באלפי תחנות, ולכן אף אחד מהם אינו מסומן על ידי `MIN_REGION_N`. הדגל קיים בעיקר עבור פילוח
   המטרופולינים ועבור מי שמריץ מחדש עם חלוקה גיאוגרפית עדינה יותר - כל עמודה מוצללת או שורה שבה
   `small_sample = True` אסור שתצוטט כממצא.

### מגבלות - הצגה כנה

- **תוויות האזורים הן רצועות קו רוחב**, שהותוו בשלב 01 מתוך ספי קואורדינטות קבועים, ולא מחוזות
  מנהליים רשמיים. תחנה הנמצאת מעט מצפון לסף משויכת לאזור שונה מזה של שכנתה בפועל. כל הנאמר במחברת
  זו יורש קירוב זה.
- **ה-betweenness הוא מקורב** (דגימת `k` מקורות בשלב קודם) ומחושב על הרכיב הקשיר הגדול ביותר בלבד,
  ולכן תחנות ברכיבים קטנים מקבלות ציון 0 ולעולם לא ייקראו קריטיות - אף שהימצאות ברכיב מבודד וזעיר
  היא כשלעצמה סוג של שבריריות.
- **זהו ניתוח טופולוגי.** תחנה בעלת betweenness גבוה אינה בהכרח תחנה בעלת נוסעים רבים; בגרף אין
  נתוני היקף נסיעות.
- **אין כאן שום קביעה סיבתית.** התוצאה אומרת שרשתות הפריפריה דלילות יותר מבחינה מבנית, אך לא מדוע,
  ולא מה תהיה עלותה של סגירה ספציפית - מחברת 05 (robustness) היא המקום שבו הסרה אכן מדומה.